# トルバプタン導入前後の体重推移
- 対象: `tolvaptan_study` の `Patient_ID` に一致する患者
- 期間: `ind_date` の前後30日（両端含む）
- 出典: `karte` の `weight` 列、および `karte_text` 内の体重記述
- 出力: 日付×体重の縦表記（患者ごと）

**区分の意味**

| 区分 | 内容 |
|---|---|
| 実測 | `weight` 列、または本文「体重/BW + 数値」の直接記載 |
| 逆算 | 本文「前日から○kg減って△kgになった」から前日分を算出 |
| 類推 | 前日差のみ記載。隣接日に基準値がある場合のみ算出 |


In [17]:
import re
import sqlite3
import unicodedata

import pandas as pd

DB_PATH = "/Users/muna/Hana_research/data/db/Hana_Research.db"
KARTE_TABLE = "karte"
STUDY_TABLE = "tolvaptan_study"
WINDOW_DAYS = 30
W_MIN, W_MAX = 20.0, 200.0   # 体重として許容する範囲(kg)。範囲外は抽出ミスとして除外

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 80)


## 1. データ読み込み・期間絞り込み

In [18]:
conn = sqlite3.connect(DB_PATH)

study = pd.read_sql(f'SELECT "Patient_ID", "ind_date" FROM {STUDY_TABLE}', conn)
study["Patient_ID"] = study["Patient_ID"].astype(str)
study["ind_date"] = pd.to_datetime(study["ind_date"], errors="coerce").dt.normalize()
n_all = study["Patient_ID"].nunique()
study = study.dropna(subset=["ind_date"]).drop_duplicates(["Patient_ID", "ind_date"])
print(f"study: {n_all}例 / ind_date有効 {study['Patient_ID'].nunique()}例")

karte = pd.read_sql(
    f'SELECT "Patient_ID", "visit_datetime", "weight", "karte_text" FROM {KARTE_TABLE}', conn
)
conn.close()
karte["Patient_ID"] = karte["Patient_ID"].astype(str)
# 「(木)」などの曜日表記を除去してからパース
karte["visit_datetime"] = (
    karte["visit_datetime"]
    .astype(str)
    .str.replace(r"\([月火水木金土日]\)", "", regex=True)
    .str.strip()
)
karte["visit_datetime"] = pd.to_datetime(
    karte["visit_datetime"], format="%Y/%m/%d %H:%M", errors="coerce"
)
n_before = len(karte)
karte = karte.dropna(subset=["visit_datetime"])
print(f"visit_datetime 変換失敗: {n_before - len(karte)}件 / {n_before}件")
karte = karte.dropna(subset=["visit_datetime"])
karte["date"] = karte["visit_datetime"].dt.normalize()

delta = pd.Timedelta(days=WINDOW_DAYS)
df = karte.merge(study, on="Patient_ID", how="inner")
df = df[(df["date"] >= df["ind_date"] - delta) & (df["date"] <= df["ind_date"] + delta)]
df = df.sort_values(["Patient_ID", "visit_datetime"]).reset_index(drop=True)
print(f"期間内カルテ: {len(df)}件 / {df['Patient_ID'].nunique()}例")


study: 214例 / ind_date有効 214例
visit_datetime 変換失敗: 7件 / 440349件
期間内カルテ: 4107件 / 214例


In [19]:
# --- 診断 ---
print("study Patient_ID 型・例:", study["Patient_ID"].dtype, study["Patient_ID"].head(3).tolist())
print("karte Patient_ID 型・例:", karte["Patient_ID"].dtype, karte["Patient_ID"].head(3).tolist())

common_ids = set(study["Patient_ID"]) & set(karte["Patient_ID"])
print("共通ID数:", len(common_ids))

print("visit_datetime NaT率:", karte["visit_datetime"].isna().mean())
print("visit_datetime 変換前サンプル:",
      pd.read_sql(f'SELECT "visit_datetime" FROM {KARTE_TABLE} LIMIT 5', sqlite3.connect(DB_PATH)))

print("ind_date サンプル:", study["ind_date"].head(3).tolist())
print("date サンプル:", karte["date"].head(3).tolist())

study Patient_ID 型・例: str ['150004', '150009', '150027']
karte Patient_ID 型・例: str ['160055', '160056', '160051']
共通ID数: 214
visit_datetime NaT率: 0.0
visit_datetime 変換前サンプル:        visit_datetime
0  2016/3/31(木) 21:05
1  2016/3/31(木) 17:40
2  2016/3/31(木) 17:04
3  2016/3/31(木) 16:52
4  2016/3/31(木) 16:28
ind_date サンプル: [Timestamp('2017-12-06 00:00:00'), Timestamp('2016-12-01 00:00:00'), Timestamp('2016-09-30 00:00:00')]
date サンプル: [Timestamp('2016-03-31 00:00:00'), Timestamp('2016-03-31 00:00:00'), Timestamp('2016-03-31 00:00:00')]


## 2. karte_text からの体重抽出
全角・半角は NFKC 正規化で統一。`体重` / `BW`（大文字小文字・全角半角不問）の後に `:` `：` `;` `；` 空白を挟んで数値が続くものを拾う。
キーワード後に数値がないものは無視。

In [20]:
def norm(s):
    return unicodedata.normalize("NFKC", s) if isinstance(s, str) else ""

KW = r"(?:体重|(?<![a-z])bw(?![a-z]))"
SEP = r"[\s:;]*"
UNIT = r"(?:kg|キロ)"

# 絶対値: 「体重 86kg」「BW：84.5」。直後が減/増の「体重 5kg減」は除外
ABS_RE = re.compile(
    rf"{KW}{SEP}(\d+(?:\.\d+)?)(?!\.?\d)(?!\s*{UNIT}?\s*(?:減|増|up|down))", re.I
)
# 相対: 「前日から5kg減」「前日比 -2kg」
REL_RE = re.compile(
    rf"前日(?:から|より|比|に比べ|と比べ)?[\s:;]*(?:約)?([+\-−±]?)(\d+(?:\.\d+)?)\s*{UNIT}?\s*"
    r"(減少|減|低下|下降|落ち|落|増加|増|上昇|up|down)?",
    re.I,
)
DOWN = ("減少", "減", "低下", "下降", "落ち", "落", "down")
TRAIL_RE = re.compile(rf"(\d+(?:\.\d+)?)\s*{UNIT}", re.I)


def ok(w):
    return w is not None and W_MIN <= w <= W_MAX


def extract(text):
    """return (abs_weight, signed_delta, evidence)
    signed_delta = 当日 - 前日 (減少は負)。"""
    t = norm(text)
    abs_w, delta_v, ev = None, None, ""

    m = REL_RE.search(t)
    if m:
        sign, num, word = m.group(1), float(m.group(2)), (m.group(3) or "").lower()
        if word or sign in ("-", "−", "+"):
            neg = word in DOWN or sign in ("-", "−")
            delta_v = -num if neg else num
            ev = t[max(0, m.start() - 5): m.end() + 25].replace("\n", " ")
            tail = TRAIL_RE.search(t[m.end(): m.end() + 25])   # 「減って60kgになった」
            if tail and ok(float(tail.group(1))):
                abs_w = float(tail.group(1))

    if abs_w is None:
        for a in ABS_RE.finditer(t):
            v = float(a.group(1))
            if ok(v):
                abs_w = v
                ev = ev or t[max(0, a.start() - 3): a.end() + 6].replace("\n", " ")
                break
    return abs_w, delta_v, ev


# 動作確認
for s in ["体重８６ｋｇ", "ＢＷ：　84.5kg", "bw 80", "体重：", "前日から2kg減って82kgになった",
          "前日比 -1kg", "体重 5kg減", "体重；63.2 ㎏"]:
    print(repr(s), "->", extract(s))


'体重８６ｋｇ' -> (86.0, None, '体重86kg')
'ＢＷ：\u300084.5kg' -> (84.5, None, 'BW: 84.5kg')
'bw 80' -> (80.0, None, 'bw 80')
'体重：' -> (None, None, '')
'前日から2kg減って82kgになった' -> (82.0, -2.0, '前日から2kg減って82kgになった')
'前日比 -1kg' -> (None, -1.0, '前日比 -1kg')
'体重 5kg減' -> (None, None, '')
'体重；63.2 ㎏' -> (63.2, None, '体重;63.2 kg')


## 3. 患者ごとの日別体重の組み立て
優先順位: 実測 > 逆算 > 類推。同日に複数ある場合は同順位内で最新の記録を採用し、値が異なれば備考に残す。

In [21]:
def build_patient(g, pid, ind):
    lo, hi = ind - delta, ind + delta
    one = pd.Timedelta(days=1)
    cand = {}      # date -> [(priority, order, weight, source, evidence)]
    deltas = {}    # date -> (signed_delta, evidence)  当日-前日

    def add(d, pr, order, w, src, ev):
        if lo <= d <= hi and ok(w):
            cand.setdefault(d, []).append((pr, order, round(w, 2), src, ev))

    for order, r in enumerate(g.itertuples(index=False)):
        d = r.date
        wc = pd.to_numeric(r.weight, errors="coerce")
        if pd.notna(wc):
            add(d, 1, order, float(wc), "実測", "weight列")
        a, dv, ev = extract(r.karte_text)
        if a is not None and dv is None:
            add(d, 1, order, a, "実測", ev)
        elif a is not None and dv is not None:
            add(d, 1, order, a, "実測", ev)
            add(d - one, 2, order, a - dv, "逆算", ev)
        elif dv is not None:
            deltas[d] = (dv, ev)

    final = {}
    for d, lst in cand.items():
        best = sorted(lst, key=lambda x: (x[0], -x[1]))[0]
        same = {x[2] for x in lst if x[0] == best[0]}
        note = f"同日複数値: {sorted(same)}" if len(same) > 1 else ""
        final[d] = [best[2], best[3], best[4], note]

    # 前日差のみの記載を、隣接日の基準値から解決
    changed = True
    while changed:
        changed = False
        for d in sorted(deltas):
            dv, ev = deltas[d]
            prev = d - one
            if d in final and prev not in final and lo <= prev <= hi:
                final[prev] = [round(final[d][0] - dv, 2), "類推", ev, "当日値-前日差で逆算"]
                changed = True
            elif prev in final and d not in final and lo <= d <= hi:
                final[d] = [round(final[prev][0] + dv, 2), "類推", ev, "前日値+前日差で算出"]
                changed = True

    rows = [
        dict(Patient_ID=pid, ind_date=ind.date(), date=d.date(),
             day_from_ind=(d - ind).days, weight_kg=v[0], 区分=v[1], 根拠=v[2], 備考=v[3])
        for d, v in sorted(final.items())
    ]
    return rows


all_rows = []
for (pid, ind), g in df.groupby(["Patient_ID", "ind_date"]):
    all_rows += build_patient(g, pid, ind)

out = pd.DataFrame(all_rows, columns=["Patient_ID", "ind_date", "date", "day_from_ind",
                                      "weight_kg", "区分", "根拠", "備考"])
print(f"体重データあり: {out['Patient_ID'].nunique()}例 / {len(out)}行")
print(out["区分"].value_counts().to_string())


体重データあり: 145例 / 734行
区分
実測    715
逆算     18
類推      1


## 4. 結果（縦表記）

In [22]:
out.head(60)

,Patient_ID,ind_date,date,day_from_ind,weight_kg,区分,根拠,備考
0,150004,2017-12-06,2017-11-08,-28,71.9,実測,/1の体重:71.9kg 11/,
1,150004,2017-12-06,2017-11-17,-19,66.0,実測,\r 体重:66kg 11/,
2,150004,2017-12-06,2017-12-01,-5,69.9,実測,\r 体重:69.90kg 前回往,
3,150009,2016-12-01,2016-11-01,-30,41.0,実測,連\r 体重41kg。下肢浮,
4,150009,2016-12-01,2016-11-07,-24,41.0,実測,。\r Bw 41kg。>洋服,
5,150009,2016-12-01,2016-11-08,-23,41.0,実測,連\r 体重41kg体重増減,
6,150009,2016-12-01,2016-11-15,-16,42.5,実測,連\r 体重42.5kg。\r 下,
7,150009,2016-12-01,2016-11-16,-15,42.5,実測,話\r 体重42.5kg服を着た,
8,150009,2016-12-01,2016-11-28,-3,41.0,実測,。\r Bw 41kg→40k,
9,150009,2016-12-01,2016-11-29,-2,39.0,実測,連\r 体重39kgkg(前,


In [23]:
# 特定患者の確認用
def show(pid):
    return out[out["Patient_ID"] == str(pid)].reset_index(drop=True)

# show("患者ID")


## 5. 体重データが取れなかった患者（要確認）

In [26]:
have = set(out["Patient_ID"])
missing = study[~study["Patient_ID"].isin(have)]
print(f"体重なし: {len(missing)}例")
missing
# --- HF=1 の患者のうち、体重測定が0件の人数 ---
conn = sqlite3.connect(DB_PATH)
hf = pd.read_sql(f'SELECT "Patient_ID", "HF" FROM {STUDY_TABLE}', conn)
conn.close()

hf["Patient_ID"] = hf["Patient_ID"].astype(str).str.strip()
hf_pos = hf[hf["HF"] == 1]["Patient_ID"]

have_weight = set(out["Patient_ID"])
hf_no_weight = hf_pos[~hf_pos.isin(have_weight)]

print(f"HF=1 対象: {len(hf_pos)}例")
print(f"HF=1 かつ体重測定なし: {len(hf_no_weight)}例")
hf_no_weight.to_frame(name="Patient_ID")

体重なし: 69例
HF=1 対象: 163例
HF=1 かつ体重測定なし: 49例


,Patient_ID
6,150068
11,151068
12,160050
18,160097
25,160260
26,160272
30,170020
32,170048
40,170173
45,170235


## 6. CSV出力

In [25]:
out.to_csv("tolvaptan_weight_long.csv", index=False, encoding="utf-8-sig")
print("saved: tolvaptan_weight_long.csv")


saved: tolvaptan_weight_long.csv


In [27]:
# --- 導入前体重（ind_date以前、直近の値）を算出 ---
pre = out[out["day_from_ind"] <= 0].copy()
pre = pre.sort_values(["Patient_ID", "date"], ascending=[True, False])
pre_weight = (
    pre.groupby("Patient_ID", as_index=False)
       .first()[["Patient_ID", "weight_kg", "date", "区分"]]
       .rename(columns={"weight_kg": "pre_weight", "date": "pre_weight_date", "区分": "pre_weight_type"})
)
print(f"導入前体重あり: {len(pre_weight)}例 / study全体: {study['Patient_ID'].nunique()}例")
pre_weight.head()
# --- tolvaptan_study テーブルへの列追加・書き込み ---
conn = sqlite3.connect(DB_PATH)
cur = conn.cursor()

existing_cols = [r[1] for r in cur.execute(f'PRAGMA table_info({STUDY_TABLE})')]
for col, typ in [("pre_weight", "REAL"), ("pre_weight_date", "TEXT")]:
    if col not in existing_cols:
        cur.execute(f'ALTER TABLE {STUDY_TABLE} ADD COLUMN "{col}" {typ}')

study_ids = pd.read_sql(f'SELECT "Patient_ID" FROM {STUDY_TABLE}', conn)
study_ids["Patient_ID"] = study_ids["Patient_ID"].astype(str).str.strip()
merged = study_ids.merge(pre_weight[["Patient_ID", "pre_weight", "pre_weight_date"]],
                          on="Patient_ID", how="left")

cur.executemany(
    f'UPDATE {STUDY_TABLE} SET "pre_weight"=?, "pre_weight_date"=? WHERE "Patient_ID"=?',
    [(r.pre_weight, str(r.pre_weight_date) if pd.notna(r.pre_weight_date) else None, r.Patient_ID)
     for r in merged.itertuples(index=False)]
)
conn.commit()
conn.close()
print("tolvaptan_study を更新しました（pre_weight, pre_weight_date）")

導入前体重あり: 97例 / study全体: 214例
tolvaptan_study を更新しました（pre_weight, pre_weight_date）
